### The University of Melbourne, School of Computing and Information Systems
# COMP90086 Computer Vision, 2025 Semester 2

## FINAL PROJECT

In [2]:
import tensorflow as tf
import pandas as pd
import os
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --- Configuration ---
IMG_HEIGHT = 320
IMG_WIDTH = 240
BATCH_SIZE = 32
BASE_DIR = "Nutrition5k"
VALIDATION_SPLIT = 0.20 # 20% for validation
RANDOM_SEED = 42 # Use a seed for reproducible splits

# --- 1. Load the full labeled dataset manifest ---
full_labels_path = os.path.join(BASE_DIR, "nutrition5k_train.csv")
df = pd.read_csv(full_labels_path)
df['ID'] = df['ID'].astype(str)




In [18]:
(64/48)*120



160.0

In [3]:
# complete train and test split (with normalised pixel values)
df = df.assign(train_image_path=BASE_DIR + "\\train\\color\\" + df["ID"].astype(str) + "\\rgb.png",
               test_image_path=BASE_DIR + "\\test\\color\\" + df["ID"].astype(str) + "\\rgb.png")

datagen = ImageDataGenerator(rescale=1./255,
                             validation_split=VALIDATION_SPLIT)

train_generator = datagen.flow_from_dataframe(
        dataframe=df,
        directory=os.getcwd(), # Root directory where image folders are located
        x_col='train_image_path', # Column in DataFrame containing image paths
        y_col='Value', # Column in DataFrame containing labels
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        class_mode='raw', # Or 'binary', 'sparse', 'input', None
        seed=RANDOM_SEED,
        subset='training' # For training data
    )


validation_generator = datagen.flow_from_dataframe(
        dataframe=df,
        directory=os.getcwd(), # Root directory where image folders are located
        x_col='train_image_path', # Column in DataFrame containing image paths
        y_col='Value', # Column in DataFrame containing labels
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        class_mode='raw', # Or 'binary', 'sparse', 'input', None
        seed=RANDOM_SEED,
        subset='validation' # For training data
    )

Found 2641 validated image filenames.
Found 660 validated image filenames.


In [4]:
# create dataset to load test dataset
df_test = pd.DataFrame({"ID": os.listdir(os.path.join(BASE_DIR, "test/color"))})
df_test = df_test.assign(test_image_path = BASE_DIR + "\\test\\color\\" + df_test["ID"].astype(str) + "\\rgb.png")

In [4]:
df_test

,ID,test_image_path
0,dish_3301,Nutrition5k\test\color\dish_3301\rgb.png
1,dish_3302,Nutrition5k\test\color\dish_3302\rgb.png
2,dish_3303,Nutrition5k\test\color\dish_3303\rgb.png
3,dish_3304,Nutrition5k\test\color\dish_3304\rgb.png
4,dish_3305,Nutrition5k\test\color\dish_3305\rgb.png
...,...,...
184,dish_3485,Nutrition5k\test\color\dish_3485\rgb.png
185,dish_3486,Nutrition5k\test\color\dish_3486\rgb.png
186,dish_3487,Nutrition5k\test\color\dish_3487\rgb.png
187,dish_3488,Nutrition5k\test\color\dish_3488\rgb.png


In [5]:
test_datagen=ImageDataGenerator(rescale=1./255.)
test_generator=test_datagen.flow_from_dataframe(
dataframe=df_test,
directory=os.getcwd(),
x_col="test_image_path",
y_col=None,
target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
batch_size=BATCH_SIZE,
seed=RANDOM_SEED,
shuffle=False,
class_mode=None)

Found 189 validated image filenames.


## attempt to build an autoencoder to train and predict

In [6]:
# attempt to build an autoencoder
train_generator_autoencoder = datagen.flow_from_dataframe(
        dataframe=df,
        directory=os.getcwd(), # Root directory where image folders are located
        x_col='train_image_path', # Column in DataFrame containing image paths
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        class_mode='input', # Or 'binary', 'sparse', 'input', None
        seed=RANDOM_SEED,
        subset='training' # For training data
    )


validation_generator_autoencoder = datagen.flow_from_dataframe(
        dataframe=df,
        directory=os.getcwd(), # Root directory where image folders are located
        x_col='train_image_path', # Column in DataFrame containing image paths
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        class_mode='input', # Or 'binary', 'sparse', 'input', None
        seed=RANDOM_SEED,
        subset='validation' # For training data
    )

test_generator_autoencoder=test_datagen.flow_from_dataframe(
        dataframe=df_test,
        directory=os.getcwd(),
        x_col="test_image_path",
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        seed=RANDOM_SEED,
        shuffle=False,
        class_mode=None
    )

Found 2641 validated image filenames.
Found 660 validated image filenames.
Found 189 validated image filenames.


In [14]:
images, labels = next(train_generator_autoencoder)

In [17]:
type(images)

numpy.ndarray

In [15]:
print("Shape of image batch:", images.shape)

Shape of image batch: (32, 320, 240, 3)


In [19]:
type(images.shape[0])

int

In [7]:
(BATCH_SIZE,IMG_HEIGHT,IMG_WIDTH,3)

(32, 320, 240, 3)

In [ ]:
encoder.layers[-3]

AttributeError: 'Conv2D' object has no attribute 'shape'

In [10]:
# first, the autoencoder portion (will need to determine how to pull out the training and validation labels from each of the generators to use for trainning the regression model)
# TODO: figure out why there is a shape mismatch when it attempts to do validation after the epoch has been trained (batch size determines part of the shape)
import numpy as np
batch_size = BATCH_SIZE

embedding_size = 16

encoder = tf.keras.Sequential(
    [
        layers.Input((BATCH_SIZE,IMG_HEIGHT,IMG_WIDTH,3)),
        layers.Flatten(),
        layers.Dense(50, activation = 'relu'),
        layers.Dense(embedding_size)
    ]
)

# Decoder
decoder = tf.keras.Sequential(
    [
        layers.Input((embedding_size,)),
        layers.Dense(50, activation='relu'),
        layers.Dense(IMG_HEIGHT*IMG_WIDTH*3),
        layers.Reshape((IMG_HEIGHT,IMG_WIDTH,3))
    ]
)

encoder = tf.keras.Sequential(
    [
        layers.Input((IMG_HEIGHT,IMG_WIDTH,3)),
        # Conv Layer 1: Learn 32 filters, downsample to 14x14
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', strides=2),
        # Conv Layer 2: Learn 64 filters, downsample to 7x7
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', strides=2),
        layers.Flatten(),
        layers.Dense(embedding_size, name="latent_space") # The compressed representation
    ],
    name="encoder"
)

# Decoder
decoder = tf.keras.Sequential(
    [
        layers.Input((embedding_size,)),
        # Upscale from embedding to the shape before flattening
        layers.Dense(7 * 7 * 64, activation='relu'),
        layers.Reshape((7, 7, 64)),
        # De-conv Layer 1: Upsample to 14x14
        layers.Conv2DTranspose(64, (3, 3), activation='relu', padding='same', strides=2),
        # De-conv Layer 2: Upsample to 28x28
        layers.Conv2DTranspose(32, (3, 3), activation='relu', padding='same', strides=2),
        # Final Conv Layer to get back to 1 channel (grayscale)
        layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')
    ],
    name="decoder"
)

# Autoencoder is encoder followed by decoder
autoencoder = tf.keras.Sequential([encoder,decoder])

# Configure the model
autoencoder.compile(optimizer='adam',
                    loss="mse",
           #loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False), #use SparseCategoricalCrossentropy because labels are integers. If the labels are one-hot representation, please use CategoricalCrossentropy loss.
           metrics=['mse'])

STEP_SIZE_TRAIN_AUTO = train_generator_autoencoder.n//train_generator_autoencoder.batch_size
STEP_SIZE_VALID_AUTO = validation_generator_autoencoder.n//validation_generator_autoencoder.batch_size

# Start training
# we use train_images as both the input and the target value for our autoencoder (no labels!)
history_autoencoder = autoencoder.fit(train_generator_autoencoder,
                                      steps_per_epoch=STEP_SIZE_TRAIN_AUTO,
                                      validation_data=validation_generator,
                                      validation_steps=STEP_SIZE_VALID_AUTO,
                                      epochs=10)

Epoch 1/10


ValueError: Dimensions must be equal, but are 320 and 28 for '{{node compile_loss/mse/sub}} = Sub[T=DT_FLOAT](data_1, sequential_7_1/decoder_1/conv2d_7_1/Sigmoid)' with input shapes: [?,320,240,3], [?,28,28,3].

In [ ]:
test_batches.labels

Evaluate the autoencoder

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history_autoencoder.history['loss'], label='Train')
plt.plot(history_autoencoder.history['val_loss'], label='Validation')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.title('Training and validating loss')
plt.legend()
plt.show()

In [ ]:
# product the training embeddings, validation embeddings, and test embeddings
train_generator_autoencoder.reset()
train_embeddings = encoder.predict(train_generator_autoencoder)
validation_generator_autoencoder.reset()
validation_embeddings = encoder.predict(validation_generator_autoencoder)
test_generator_autoencoder.reset()
test_embeddings = encoder.predict(test_generator_autoencoder)

In [7]:
# build basic regression model
model = tf.keras.Sequential(
    [
        layers.Input((IMG_HEIGHT, IMG_WIDTH, 3)),
        
        layers.Conv2D(8, (5, 5), activation='relu'), # fill in
        layers.MaxPooling2D((2, 2)), # fill in
        
        layers.Flatten(),
        layers.Dense(10, activation="relu"),
        layers.Dense(1, activation="linear")
    ], 
)

model.compile(optimizer='adam', loss='mse', metrics=['mse'])

In [17]:
STEP_SIZE_TRAIN = train_generator.n//train_generator.batch_size
STEP_SIZE_VALID = validation_generator.n//validation_generator.batch_size

model.fit(train_generator,
          steps_per_epoch=STEP_SIZE_TRAIN,
          validation_data=validation_generator,
          validation_steps=STEP_SIZE_VALID,
          epochs=20
)


Epoch 1/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 51s 622ms/step - loss: 16333.6514 - mse: 16333.6514 - val_loss: 18809.3867 - val_mse: 18809.3867
Epoch 2/20
 1/82 ━━━━━━━━━━━━━━━━━━━━ 21s 261ms/step - loss: 429735.1875 - mse: 429735.1875

c:\Users\nares\anaconda3\envs\CV\lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


82/82 ━━━━━━━━━━━━━━━━━━━━ 8s 98ms/step - loss: 429735.1875 - mse: 429735.1875 - val_loss: 19897.7402 - val_mse: 19897.7402
Epoch 3/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 50s 607ms/step - loss: 28186.5703 - mse: 28186.5703 - val_loss: 20509.4023 - val_mse: 20509.4023
Epoch 4/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 7s 88ms/step - loss: 12755.5283 - mse: 12755.5283 - val_loss: 19218.4492 - val_mse: 19218.4492
Epoch 5/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 48s 589ms/step - loss: 15033.2031 - mse: 15033.2031 - val_loss: 16551.2500 - val_mse: 16551.2500
Epoch 6/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 8s 90ms/step - loss: 11581.6562 - mse: 11581.6562 - val_loss: 17622.5977 - val_mse: 17622.5977
Epoch 7/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 46s 559ms/step - loss: 13969.6758 - mse: 13969.6758 - val_loss: 15757.3770 - val_mse: 15757.3770
Epoch 8/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 7s 88ms/step - loss: 4708.4004 - mse: 4708.4004 - val_loss: 15937.9160 - val_mse: 15937.9160
Epoch 9/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 46s 561ms/step - loss: 8377.0410 - m

In [ ]:
# validation
STEP_SIZE_TEST=test_generator.n//test_generator.batch_size
model.evaluate(validation_generator)

21/21 ━━━━━━━━━━━━━━━━━━━━ 7s 344ms/step - loss: 14278.6387 - mse: 14278.6387


[14662.9150390625, 14662.9150390625]

In [21]:
test_generator.reset()
preds=model.predict(test_generator)

6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 376ms/step


In [22]:
df_test = df_test.assign(Value=preds)

In [23]:
df_test

,ID,test_image_path,Value
0,dish_3301,Nutrition5k\test\color\dish_3301\rgb.png,853.766052
1,dish_3302,Nutrition5k\test\color\dish_3302\rgb.png,152.479446
2,dish_3303,Nutrition5k\test\color\dish_3303\rgb.png,91.166473
3,dish_3304,Nutrition5k\test\color\dish_3304\rgb.png,161.127014
4,dish_3305,Nutrition5k\test\color\dish_3305\rgb.png,360.575531
...,...,...,...
184,dish_3485,Nutrition5k\test\color\dish_3485\rgb.png,132.760986
185,dish_3486,Nutrition5k\test\color\dish_3486\rgb.png,0.033012
186,dish_3487,Nutrition5k\test\color\dish_3487\rgb.png,378.267456
187,dish_3488,Nutrition5k\test\color\dish_3488\rgb.png,165.710480


In [24]:
df_submit = df_test.drop("test_image_path", axis=1)

In [25]:
df_submit

,ID,Value
0,dish_3301,853.766052
1,dish_3302,152.479446
2,dish_3303,91.166473
3,dish_3304,161.127014
4,dish_3305,360.575531
...,...,...
184,dish_3485,132.760986
185,dish_3486,0.033012
186,dish_3487,378.267456
187,dish_3488,165.710480


In [ ]:
# save copy to csv
df_submit.to_csv("iteration_autoencoder_submission.csv")